In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
import requests
from bs4 import BeautifulSoup

In [21]:
browser = webdriver.Chrome()
browser.get('https://www.amazon.in/gp/browse.html?node=1968120031&ref_=nav_em_sbc_mfashion_tshirts_0_2_10_3')
input_search = browser.find_element(By.ID, 'twotabsearchtextbox')
search_button = browser.find_element(By.XPATH, "(//input[@type='submit'])[1]")
# send the input to the webpage
input_search.send_keys("cotton half sleeve")
sleep(1)
search_button.click()
products = []

for i in range(6):
    print('Scraping page', i+1)
    
    # Scraping product title
    titles = browser.find_elements(By.XPATH, "//span[@class='a-size-large product-title-word-break']")
    title_texts = [title.text.strip() for title in titles]
    
    # Scraping product prices
    prices = browser.find_elements(By.XPATH, "//span[@class='a-price-whole']")
    price_values = [price.text.strip() for price in prices]
    
    # Scraping reviews
    reviews = browser.find_elements(By.XPATH, "//h3[@data-hook='dp-local-reviews-header']")
    review_texts = [review.text.strip() for review in reviews]
    
    # Adding scraped data to the products list
    for title, price, review in zip(title_texts, price_values, review_texts):
        products.append({"Title": title, "Price": price, "Reviews": review})
    
    next_button = browser.find_element(By.XPATH, "//a[@class='s-pagination-item s-pagination-next s-pagination-button s-pagination-separator']")
    next_button.click()
    sleep(5)
# Print the scraped product data
for product in products:
    print(product)    
import concurrent.futures
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException, TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)

def scrape_color_images(shoe):
    try:
        # Find all image elements for the current color
        image_elements = shoe.find_elements(By.XPATH, ".//img[@class='imgSwatch']")
        # Extract image URLs
        images = [image.get_attribute("src") for image in image_elements]
    except (NoSuchElementException, StaleElementReferenceException, TimeoutException):
        logging.exception("Failed to extract color images")
        images = []
    return images

def scrape_color_reviews(shoe):
    try:
        # Find all review elements for the current color
        review_elements = shoe.find_elements(By.XPATH, ".//div[@data-hook='review']")
        # Extract reviews
        reviews = []
        for review_element in review_elements:
            try:
                rating_element = review_element.find_element(By.XPATH, ".//i[@data-hook='review-star-rating']")
                rating = rating_element.find_element(By.XPATH, ".//span[@class='a-icon-alt']").text.split()[0]
            except NoSuchElementException:
                rating = "No rating"
            try:
                text_element = review_element.find_element(By.XPATH, ".//div[@data-hook='review-collapsed']")
                text = text_element.text.strip()
            except NoSuchElementException:
                text = "No review text available"
            reviews.append({"Rating": rating, "Review": text})
    except (NoSuchElementException, StaleElementReferenceException, TimeoutException):
        logging.exception("Failed to extract color reviews")
        reviews = []
    return reviews

def scrape_color_rating(shoe):
    try:
        # Find the overall rating for the current color
        rating_element = shoe.find_element(By.XPATH, ".//span[@class='a-icon-alt']")
        rating = rating_element.text.split()[0]
    except (NoSuchElementException, StaleElementReferenceException, TimeoutException):
        logging.exception("Failed to extract color rating")
        rating = "No rating"
    return rating

def scrape_product(shoe):
    try:
        name_element = WebDriverWait(shoe, 20).until(EC.presence_of_element_located((By.XPATH, ".//span[@class='a-size-base-plus a-color-base a-text-normal']")))
        name = name_element.text
    except (NoSuchElementException, StaleElementReferenceException, TimeoutException):
        logging.exception("Failed to extract product name")
        name = "No name available"

    try:
        product_id = shoe.get_attribute("data-asin")
    except (NoSuchElementException, StaleElementReferenceException, TimeoutException):
        logging.exception("Failed to extract product ID")
        product_id = ""

    try:
        price_element = shoe.find_element(By.XPATH, ".//span[@class='a-price-whole']")
        price = price_element.text if price_element else "0"
    except (NoSuchElementException, StaleElementReferenceException, TimeoutException):
        logging.exception("Failed to extract product price")
        price = "No price available"

    try:
        product_url_element = shoe.find_element(By.XPATH, ".//a[@class='a-link-normal s-no-outline']")
        product_url = product_url_element.get_attribute("href") if product_url_element else ""
    except (NoSuchElementException, StaleElementReferenceException, TimeoutException):
        logging.exception("Failed to extract product URL")
        product_url = ""

    try:
        brand_element = shoe.find_element(By.XPATH, ".//span[@class='a-size-base-plus a-color-base']")
        brand = brand_element.text.strip()
    except (NoSuchElementException, StaleElementReferenceException, TimeoutException):
        logging.exception("Failed to extract product description")
        brand = "No description available"

    # Color scraping logic
    colors = []
    try:
        color_elements = shoe.find_elements(By.XPATH, ".//li[@class='swatchAvailable']")
        for color_element in color_elements:
            color_name = color_element.get_attribute("title")
            color_element.click()
            time.sleep(1)  # Wait for the page to load after selecting color
            color_images = scrape_color_images(shoe)
            color_reviews = scrape_color_reviews(shoe)
            color_rating = scrape_color_rating(shoe)
            colors.append({
                "color_name": color_name,
                "images": color_images,
                "reviews": color_reviews,
                "rating": color_rating
            })
    except (NoSuchElementException, StaleElementReferenceException, TimeoutException):
        logging.exception("Failed to extract colors")

    return {
        "shoe_name": name,
        "product_id": product_id,
        "price": price,
        "product_url": product_url,
        "brand_name": brand,
        "colors": colors
    }

def main():
    # Use the Firefox driver
    driver = webdriver.Firefox()

    try:
        driver.get("https://www.amazon.in")
        driver.maximize_window()

        search_box = driver.find_element(By.ID, "twotabsearchtextbox")
        search_box.clear()
        search_box.send_keys("shoes for women")
        driver.find_element(By.ID, "nav-search-submit-button").click()

        # Add a delay to allow the page to load
        time.sleep(2)

        # Adjusted XPath expression to capture all product elements
        shoes = driver.find_elements(By.XPATH, '//div[@data-component-type="s-search-result"]')

        data = []

        # Concurrently scrape product information
        with concurrent.futures.ThreadPoolExecutor() as executor:
            results = executor.map(scrape_product, shoes)

        # Collect results
        for result in results:
            data.append(result)

        print(json.dumps(data, indent=4))

    finally:
        driver.quit()

if  '_name_' == "_main_":
    main()
print("hysam")        

Scraping page 1
Scraping page 2
Scraping page 3
Scraping page 4
Scraping page 5


AttributeError: 'NoneType' object has no attribute 'strip'

In [3]:
browser = webdriver.Chrome()
browser.get('https://www.amazon.in/gp/browse.html?node=1968120031&ref_=nav_em_sbc_mfashion_tshirts_0_2_10_3')
input_search = browser.find_element(By.ID, 'twotabsearchtextbox')
search_button = browser.find_element(By.XPATH, "(//input[@type='submit'])[1]")
# send the input to the webpage
input_search.send_keys("cotton half sleeve")
sleep(1)
search_button.click()
products = []
browser.get('https://www.amazon.in/Hmkm-Casual-Stylish-Printed-Medium/dp/B0C3MMM6RM/ref=sr_1_97?crid=2U4ID756ZFZEE&dib=eyJ2IjoiMSJ9.F3q8Vgn5XIH070pW71mJWmlw4wB09MARpDXQUg9SPHKzTzGsL9vDsZuPIi2lKvku4BIVdC_lQvfb5YHNqy0RVvGd0JGp97_z02Fh3EKpei1CwW2P-b5fMfo2N9BBueyFIZBQd7gGOKnTLpJH8LZ91q0_YCaXn3NGz_GNA4d1Y7C4sb5g7sf8IdK98gJF8JynucbqNk8vSw2CzfG15DwBUbrUKwBWx-_-x250T7nuE9LCBTD0Yq79CXK7jV11T4WTeG83XQNAGjAkQc7bceOWOhUCgTDnPXA6kXLdT2tLphI.Wk9-e4FlBdQhgU4mtrG5e04o-s2gw8HEVk2BuB3mwvk&dib_tag=se&keywords=cotton+half+sleeve&qid=1709677444&sprefix=cotton+half+sleeve%2Caps%2C219&sr=8-97-spons&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGZfbmV4dA&psc=1')
title_element = browser.find_element(By.XPATH,"/html/body/div[2]/div[1]/div/div[7]/div[1]/div[2]/div[2]/div/div/div[1]/div[1]/div[2]/div[1]/div/h1/span")


title_text = title_element.text


price_element = browser.find_element(By.XPATH,'/html/body/div[2]/div[1]/div/div[7]/div[1]/div[2]/div[2]/div/div/div[1]/div[3]/div/div/div[4]/div[1]/span[3]/span[2]/span[2]')


price = price_element.text



image_element =browser.find_element(By.XPATH,'/html/body/div[2]/div[1]/div/div[7]/div[1]/div[2]/div[1]/div[1]/div[1]/div/div/div[2]/div[1]/div[1]/ul/li[1]/span/span/div/img')

image_url = image_element.get_attribute('src')
d=browser.find_element(By.XPATH,'/html/body/div[2]/div[1]/div/div[7]/div[1]/div[2]/div[2]/div/div/div[1]/div[31]/div/div[1]')
d=d.text
print(f"The extracted image URL is: {image_url}")
print(f"The extracted price is: {price}")


print("Title:", title_text)
print('des',d)



The extracted image URL is: https://m.media-amazon.com/images/I/71t4AzlfXDL._SY679_.jpg
The extracted price is: 499
Title: Hmkm Casual Shirt Men Stylish Shirt || Men Printed Shirt
des Product details
Material composition
Layacara
Pattern
Printed
Fit type
Relaxed Fit
Sleeve type
Half Sleeve
Length
Standard Length
Neck style
Dom
Country of Origin
India
About this item
Care Instructions: Dry Clean Only
Style - Enhance your look by wearing this Casual Stylish Men's shirt, Team it with a pair of tapered denims Or Solid Chinos and Loafers for a fun Smart Casual look


In [5]:
browser = webdriver.Chrome()
browser.get('https://www.amazon.in/gp/browse.html?node=1968120031&ref_=nav_em_sbc_mfashion_tshirts_0_2_10_3')
input_search = browser.find_element(By.ID, 'twotabsearchtextbox')
search_button = browser.find_element(By.XPATH, "(//input[@type='submit'])[1]")
# send the input to the webpage
input_search.send_keys("cotton half sleeve")
sleep(1)
search_button.click()
products = []
browser.get('https://www.amazon.in/Hmkm-Casual-Stylish-Printed-Medium/dp/B0C3MMM6RM/ref=sr_1_97?crid=2U4ID756ZFZEE&dib=eyJ2IjoiMSJ9.F3q8Vgn5XIH070pW71mJWmlw4wB09MARpDXQUg9SPHKzTzGsL9vDsZuPIi2lKvku4BIVdC_lQvfb5YHNqy0RVvGd0JGp97_z02Fh3EKpei1CwW2P-b5fMfo2N9BBueyFIZBQd7gGOKnTLpJH8LZ91q0_YCaXn3NGz_GNA4d1Y7C4sb5g7sf8IdK98gJF8JynucbqNk8vSw2CzfG15DwBUbrUKwBWx-_-x250T7nuE9LCBTD0Yq79CXK7jV11T4WTeG83XQNAGjAkQc7bceOWOhUCgTDnPXA6kXLdT2tLphI.Wk9-e4FlBdQhgU4mtrG5e04o-s2gw8HEVk2BuB3mwvk&dib_tag=se&keywords=cotton+half+sleeve&qid=1709677444&sprefix=cotton+half+sleeve%2Caps%2C219&sr=8-97-spons&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGZfbmV4dA&psc=1')
#title_element = browser.find_element(By.XPATH,"/html/body/div[2]/div[1]/div/div[7]/div[1]/div[2]/div[2]/div/div/div[1]/div[1]/div[2]/div[1]/div/h1/span")











In [8]:
from bs4 import BeautifulSoup
from bs4 import BeautifulSoup


product_info_list = []
price_info_list = []
brand_info_list = []
image_info_list = []
details_info_list = []
reviews_info_list = []
ratings_info_list = []
model_numbers = []

def scrape_product_data(soup):
    # Product Title
    product_title_element = soup.find('span', {'id': 'productTitle'})
    product_title = product_title_element.text.strip() if product_title_element else None
    product_info_list.append(("Product Title", product_title))

    # Brand Name
    brand_element = soup.find('a', {'id': 'bylineInfo'})
    brand_name = brand_element.text.replace('Brand: ', '').strip() if brand_element else None
    brand_info_list.append(("Brand Name", brand_name))

    # Full Price
    price_whole_element = soup.find('span', {'class': 'a-price-whole'})
    price_decimal_element = soup.find('span', {'class': 'a-price-decimal'})
    full_price = f"{price_whole_element.text.strip()}.{price_decimal_element.text.strip()}" if price_whole_element and price_decimal_element else None
    price_info_list.append(("Full Price", full_price))

    # Image URL
    img_element = soup.find('img', {'id': 'landingImage'})
    image_url = img_element['src'] if img_element and 'src' in img_element.attrs else None
    image_info_list.append(("Image URL", image_url))

    deet = []

    # Product Details
    product_details_container = soup.find('div', {'id': 'productFactsDesktopExpander'})
    if product_details_container:
        product_details_title = product_details_container.find('h3', {'class': 'product-facts-title'}).text.strip()
        deet.append(f"Product Details Title: {product_details_title}")

        details = product_details_container.find_all('div', {'class': 'a-fixed-left-grid'})
        for detail in details:
            detail_title = detail.find('span', {'style': 'font-weight: 600;'}).text.strip()
            detail_value = detail.find('span', {'style': 'font-weight: 400;'}).text.strip()
            deet.append(f"{detail_title}: {detail_value}")

        # Extract the description and append as a single string
        description_title = product_details_container.find('h3', {'class': 'product-facts-title'}, string='About this item')
        description_text = ''  # Initialize description_text before the condition
        if description_title:
            description_items = product_details_container.find_all('li', {'class': 'a-list-item'})
            description_text = ' '.join(item.text.strip() for item in description_items)

        deet.append(f"Description: {description_text}")

    d_combined = ' '.join(deet)
    details_info_list.append(("description", d_combined))

    #model_number_span = soup.find('span', text='B09D8SJTKQ')

    # Check if the element was found before trying to access its text attribute
    #if model_number_span:
        # Extract the model number text
     #   model_number = model_number_span.text.strip()
        # Append to the list
      #  model_numbers = [model_number]

    review_containers = soup.find_all('div', {'data-hook': 'review-collapsed'})
    reviews = []

    if review_containers:
        for index, review_container in enumerate(review_containers, start=1):
            review_text = review_container.text.strip()
            reviews.append(review_text)

    # Optional: Join reviews into a single string
    reviews_combined = ' '.join(reviews)
    reviews_info_list.append(("Reviews", reviews_combined))

    # Ratings
    ratings_element = soup.find('span', {'class': 'a-size-base a-color-base'})
    ratings = ratings_element.text.strip() if ratings_element else None
    ratings_info_list.append(("Ratings", ratings))

    return {
        "Product Info": product_info_list,
        "Brand Info": brand_info_list,
        "Price Info": price_info_list,
        "Image Info": image_info_list,
        "Details Info": details_info_list,
        "Reviews Info": reviews_info_list,
        "Ratings Info": ratings_info_list,
        #"Model Numbers": model_numbers
    }








# Example usage:
# Call this function with your BeautifulSoup object







driver = webdriver.Chrome()
driver.maximize_window()

driver.get('https://www.amazon.com/s?i=specialty-aps&bbn=16225021011&rh=n%3A7141123011%2Cn%3A16225021011%2Cn%3A679182011&ref=nav_em__nav_desktop_sa_intl_shoes_0_2_15_3')
first_click =driver.find_element(By.XPATH,'/html/body/div/div/a/img')
first_click.click()



time.sleep(1)

driver.get('https://www.amazon.com/s?i=fashion-mens-intl-ship&bbn=16225019011&rh=n%3A7141123011%2Cn%3A16225019011%2Cn%3A1040658&page=2&qid=1709672244&ref=sr_pg_1')




#driver.get('https://www.amazon.com/Legendary-Whitetails-Flannels-Barnwood-Medium/dp/B07FNZMVP7/ref=sr_1_1?dib=eyJ2IjoiMSJ9.gYNtHunUC7rkyiJOggT3DEXXoktxi07lQPKUKV0UT7Xb0KHKLyoWWDbRyRjSxn-0xa_PbW0JGrLl1hcrPdwbggRNCirjM3iNdacCc-b0SDIjEy7F8h7lA1tTNZn_eGIhrF6h-1cRnijBthqBrH_DUNEpEsl-_Q_6_ItQM4l4WOZJSqBIqrQ9Y9LeemAAvj_2XBXj7CeTiUStMYqafNyczF_O7T88PSPVHYRqrCb3vT5bMwVzJNpm1K68jMncUnfQXlRb3DRn3TykvHSeJjpgvMGKdLImsefm_mNjXoXmhmw.cyGV9B8MRSxZgkz00P3qcaonKY-DUnIKoDbOvHz66KI&dib_tag=se&qid=1709666980&s=fashion-mens-intl-ship&sr=1-1-spons&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGZfYnJvd3Nl&psc=1')

# Get the page source
html = driver.page_source

# Use Beautiful Soup to parse the HTML
soup = BeautifulSoup(html, 'html.parser')

# Find all links (anchors) in the HTML
all_links = soup.find_all('a')

sspa_links = []

# Extract and store the href attribute from each link
for link in all_links:
    href = link.get('href')
    if href and href.startswith('/sspa/click'):
        sspa_links.append(href)

# Print the list of links
#print(sspa_links)

# Close the browser
driver.quit()



# Loop through each link and open it in the Chrome browser
for link in sspa_links:
    driver = webdriver.Chrome()
    driver.maximize_window()

    driver.get('https://www.amazon.com/s?i=specialty-aps&bbn=16225021011&rh=n%3A7141123011%2Cn%3A16225021011%2Cn%3A679182011&ref=nav_em__nav_desktop_sa_intl_shoes_0_2_15_3')
    first_click =driver.find_element(By.XPATH,'/html/body/div/div/a/img')
    first_click.click()

    

    time.sleep(1)
    full_link = 'https://www.amazon.com' + link  # Assuming the links are relative, add the base URL
    driver.get(full_link)
    html = driver.page_source

    # Use Beautiful Soup to parse the HTML
    soup = BeautifulSoup(html, 'html.parser')

    result = scrape_product_data(soup)


    driver.quit()
    time.sleep(1)
# Close the browser after all links have been opened



NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"/html/body/div/div/a/img"}
  (Session info: chrome=122.0.6261.95); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF7F3E4AD22+56930]
	(No symbol) [0x00007FF7F3DBF622]
	(No symbol) [0x00007FF7F3C742E5]
	(No symbol) [0x00007FF7F3CB98ED]
	(No symbol) [0x00007FF7F3CB9A2C]
	(No symbol) [0x00007FF7F3CFA967]
	(No symbol) [0x00007FF7F3CDBCDF]
	(No symbol) [0x00007FF7F3CF81E2]
	(No symbol) [0x00007FF7F3CDBA43]
	(No symbol) [0x00007FF7F3CAD438]
	(No symbol) [0x00007FF7F3CAE4D1]
	GetHandleVerifier [0x00007FF7F41C6AAD+3709933]
	GetHandleVerifier [0x00007FF7F421FFED+4075821]
	GetHandleVerifier [0x00007FF7F421817F+4043455]
	GetHandleVerifier [0x00007FF7F3EE9756+706710]
	(No symbol) [0x00007FF7F3DCB8FF]
	(No symbol) [0x00007FF7F3DC6AE4]
	(No symbol) [0x00007FF7F

# NOW MAKING DATA FRAME TO CHECK DATA

In [10]:
import pandas as pd
# Creating a DataFrame
data = {
    'Product Info': product_info_list,
    'Price Info': price_info_list,
    'Brand Info': brand_info_list,
    'Image Info': image_info_list,
    'Details Info': details_info_list,
    'Reviews Info': reviews_info_list,
    'Ratings Info': ratings_info_list
}

df = pd.DataFrame(data)

# Displaying the DataFrame
print(df)

                                         Product Info           Price Info  \
0   (Product Title, Pioneer Hi Vis Long Sleeved Co...  (Full Price, 53...)   
1   (Product Title, Pioneer Hi Vis Long Sleeved Co...  (Full Price, 53...)   
2   (Product Title, Pioneer Hi Vis Long Sleeved Co...  (Full Price, 53...)   
3   (Product Title, Legendary Whitetails Men's Buc...  (Full Price, 29...)   
4   (Product Title, Legendary Whitetails Men's Buc...  (Full Price, 29...)   
5   (Product Title, Legendary Whitetails Men's Buc...  (Full Price, 29...)   
6   (Product Title, Legendary Whitetails Men's Buc...  (Full Price, 29...)   
7   (Product Title, Lee Men's Big & Tall Extreme M...  (Full Price, 39...)   
8   (Product Title, Lee Men's Big & Tall Extreme M...  (Full Price, 39...)   
9   (Product Title, Lee Men's Big & Tall Extreme M...  (Full Price, 39...)   
10  (Product Title, Lee Men's Big & Tall Extreme M...  (Full Price, 39...)   
11  (Product Title, Lee Men's Big & Tall Extreme M...  (Full Pri

In [11]:
df

,Product Info,Price Info,Brand Info,Image Info,Details Info,Reviews Info,Ratings Info
0,"(Product Title, Pioneer Hi Vis Long Sleeved Co...","(Full Price, 53...)","(Brand Name, Visit the Pioneer Store)","(Image URL, https://m.media-amazon.com/images/...","(description, )","(Reviews, )","(Ratings, $53.75)"
1,"(Product Title, Pioneer Hi Vis Long Sleeved Co...","(Full Price, 53...)","(Brand Name, Visit the Pioneer Store)","(Image URL, https://m.media-amazon.com/images/...","(description, )","(Reviews, )","(Ratings, $53.75)"
2,"(Product Title, Pioneer Hi Vis Long Sleeved Co...","(Full Price, 53...)","(Brand Name, Visit the Pioneer Store)","(Image URL, https://m.media-amazon.com/images/...","(description, )","(Reviews, )","(Ratings, $53.75)"
3,"(Product Title, Legendary Whitetails Men's Buc...","(Full Price, 29...)","(Brand Name, Visit the Legendary Whitetails St...","(Image URL, https://m.media-amazon.com/images/...","(description, )","(Reviews, Very soft, true to fit. Nice shirt. ...","(Ratings, $29.50)"
4,"(Product Title, Legendary Whitetails Men's Buc...","(Full Price, 29...)","(Brand Name, Visit the Legendary Whitetails St...","(Image URL, https://m.media-amazon.com/images/...","(description, )","(Reviews, Very soft, true to fit. Nice shirt. ...","(Ratings, $29.50)"
5,"(Product Title, Legendary Whitetails Men's Buc...","(Full Price, 29...)","(Brand Name, Visit the Legendary Whitetails St...","(Image URL, https://m.media-amazon.com/images/...","(description, )","(Reviews, Very soft, true to fit. Nice shirt. ...","(Ratings, $29.50)"
6,"(Product Title, Legendary Whitetails Men's Buc...","(Full Price, 29...)","(Brand Name, Visit the Legendary Whitetails St...","(Image URL, https://m.media-amazon.com/images/...","(description, )","(Reviews, Very soft, true to fit. Nice shirt. ...","(Ratings, $29.50)"
7,"(Product Title, Lee Men's Big & Tall Extreme M...","(Full Price, 39...)","(Brand Name, Visit the Lee Store)","(Image URL, https://m.media-amazon.com/images/...","(description, Product Details Title: Product d...","(Reviews, I was hesitant to purchase jeans onl...","(Ratings, 4.7)"
8,"(Product Title, Lee Men's Big & Tall Extreme M...","(Full Price, 39...)","(Brand Name, Visit the Lee Store)","(Image URL, https://m.media-amazon.com/images/...","(description, Product Details Title: Product d...","(Reviews, I was hesitant to purchase jeans onl...","(Ratings, 4.7)"
9,"(Product Title, Lee Men's Big & Tall Extreme M...","(Full Price, 39...)","(Brand Name, Visit the Lee Store)","(Image URL, https://m.media-amazon.com/images/...","(description, Product Details Title: Product d...","(Reviews, I was hesitant to purchase jeans onl...","(Ratings, 4.7)"


STORING IN .JSON FILE

In [12]:
df.to_json('Ayera.json', orient='records', lines=True)


In [10]:
browser = webdriver.Chrome()
browser.get('https://www.amazon.in/gp/browse.html?node=1968120031&ref_=nav_em_sbc_mfashion_tshirts_0_2_10_3')
input_search = browser.find_element(By.ID, 'twotabsearchtextbox')
search_button = browser.find_element(By.XPATH, "(//input[@type='submit'])[1]")
# send the input to the webpage
input_search.send_keys("cotton half sleeve")
sleep(1)
search_button.click()
products = []
response = requests.get('https://www.amazon.in/gp/browse.html?node=1968120031&ref_=nav_em_sbc_mfashion_tshirts_0_2_10_3')
html_content = response.content


soup = BeautifulSoup(html_content, 'html.parser')


links = soup.find_all('a')

for link in links:
    href = link.get('href')
    if href:
        print(href)

/ref=nav_logo
/customer-preferences/edit?ie=UTF8&preferencesReturnUrl=%2F&ref_=topnav_lang
https://www.amazon.in/ap/signin?openid.pape.max_auth_age=0&openid.return_to=https%3A%2F%2Fwww.amazon.in%2FAmazon-Custom%2Fb%2F%3F_encoding%3DUTF8%26ie%3DUTF8%26node%3D32615889031%26ref_%3Dnav_ya_signin&openid.identity=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.assoc_handle=inflex&openid.mode=checkid_setup&openid.claimed_id=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&openid.ns=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0
/gp/css/order-history?ref_=nav_orders_first
https://www.amazon.in/gp/cart/view.html?ref_=nav_cart
/gp/site-directory?ref_=nav_em_js_disabled
/fresh?ref_=nav_cs_fresh
/minitv?ref_=nav_avod_desktop_topnav
/b/32702023031?node=32702023031&ld=AZINSOANavDesktop_T3&ref_=nav_cs_sell_T3
/gp/bestsellers/?ref_=nav_cs_bestsellers
/mobile-phones/b/?ie=UTF8&node=1389401031&ref_=nav_cs_mobiles
/deals?ref_=nav_cs_gb
/electronics/b/?ie=UTF8&node=97641903

In [22]:
import requests
from bs4 import BeautifulSoup


driver = webdriver.Chrome()
driver.maximize_window()

driver.get('https://www.amazon.in/Hmkm-Casual-Stylish-Printed-Medium/dp/B0C3MMM6RM/ref=sr_1_97?crid=2U4ID756ZFZEE&dib=eyJ2IjoiMSJ9.F3q8Vgn5XIH070pW71mJWmlw4wB09MARpDXQUg9SPHKzTzGsL9vDsZuPIi2lKvku4BIVdC_lQvfb5YHNqy0RVvGd0JGp97_z02Fh3EKpei1CwW2P-b5fMfo2N9BBueyFIZBQd7gGOKnTLpJH8LZ91q0_YCaXn3NGz_GNA4d1Y7C4sb5g7sf8IdK98gJF8JynucbqNk8vSw2CzfG15DwBUbrUKwBWx-_-x250T7nuE9LCBTD0Yq79CXK7jV11T4WTeG83XQNAGjAkQc7bceOWOhUCgTDnPXA6kXLdT2tLphI.Wk9-e4FlBdQhgU4mtrG5e04o-s2gw8HEVk2BuB3mwvk&dib_tag=se&keywords=cotton+half+sleeve&qid=1709677444&sprefix=cotton+half+sleeve%2Caps%2C219&sr=8-97-spons&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGZfbmV4dA&psc=1')
first_click =driver.find_element(By.XPATH,'/html/body/div/div/a/img')
first_click.click()



input_search =driver.find_element(By.XPATH,'/html/body/div[1]/header/div/div[1]/div[2]/div/form/div[2]/div[1]/input')
input_search.send_keys("Boys Fashion")

first_click =driver.find_element(By.XPATH,'/html/body/div[1]/header/div/div[1]/div[2]/div/form/div[3]/div/span/input')
first_click.click()
driver.get('https://www.amazon.com/s?i=specialty-aps&bbn=16225018011&rh=n%3A7141123011%2Cn%3A16225018011%2Cn%3A1040660&ref=nav_em__nav_desktop_sa_intl_clothing_0_2_12_2')

page_source = driver.page_source


url = 'https://www.amazon.in/Blueeagle-Graphic-Oversized-Pink-XS/dp/B0CVH5L4S3/ref=sr_1_1_sspa?crid=3R7MEVI23R8YF&dib=eyJ2IjoiMSJ9._KFttclUCXBrw4r9hlt_NqVB86HfqS4GmoOGiTHNWSmAZ-JFrDaKOweFP_V6I75iD09-RvQjWKNTvLWKD0jZMjKWmlYOqIOM1BZbpBPiGgCtzh64aTqbMWtx1dQLgASddCUBZ_CjyTkYBxrpRqrAe_TppbtAvXgTVtmuffhwsPeM--qpP41RM7CKl-qPV48vNffYiGSzE1jzLUjCLm5ARmmgSWAYGhIqoHhZ2fV-hCen-sfYgrnF_LtEKpw8NxihOengcUAXWO-jY5WCX79hkOHgenK0EFQkPxy-Ll1T6nY.Iul9J1kMx_-Buo3Oz3y6x1OvFfhqR0CcX7MWPO_SPxg&dib_tag=se&keywords=cotton+half+sleeve&qid=1709676309&sprefix=cotton+half+sleeve%2Caps%2C464&sr=8-1-spons&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGY&psc=1'

response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Find the title h1 element
soup


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"/html/body/div/div/a/img"}
  (Session info: chrome=122.0.6261.95); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF669F2AD22+56930]
	(No symbol) [0x00007FF669E9F622]
	(No symbol) [0x00007FF669D542E5]
	(No symbol) [0x00007FF669D998ED]
	(No symbol) [0x00007FF669D99A2C]
	(No symbol) [0x00007FF669DDA967]
	(No symbol) [0x00007FF669DBBCDF]
	(No symbol) [0x00007FF669DD81E2]
	(No symbol) [0x00007FF669DBBA43]
	(No symbol) [0x00007FF669D8D438]
	(No symbol) [0x00007FF669D8E4D1]
	GetHandleVerifier [0x00007FF66A2A6AAD+3709933]
	GetHandleVerifier [0x00007FF66A2FFFED+4075821]
	GetHandleVerifier [0x00007FF66A2F817F+4043455]
	GetHandleVerifier [0x00007FF669FC9756+706710]
	(No symbol) [0x00007FF669EAB8FF]
	(No symbol) [0x00007FF669EA6AE4]
	(No symbol) [0x00007FF66

In [23]:
input_search = browser.find_element(By.ID, 'twotabsearchtextbox')
search_button = browser.find_element(By.XPATH, "(//input[@type='submit'])[1]")

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=122.0.6261.95)
Stacktrace:
	GetHandleVerifier [0x00007FF669F2AD22+56930]
	(No symbol) [0x00007FF669E9F622]
	(No symbol) [0x00007FF669D542E5]
	(No symbol) [0x00007FF669D31D4C]
	(No symbol) [0x00007FF669DC23F7]
	(No symbol) [0x00007FF669DD7891]
	(No symbol) [0x00007FF669DBBA43]
	(No symbol) [0x00007FF669D8D438]
	(No symbol) [0x00007FF669D8E4D1]
	GetHandleVerifier [0x00007FF66A2A6AAD+3709933]
	GetHandleVerifier [0x00007FF66A2FFFED+4075821]
	GetHandleVerifier [0x00007FF66A2F817F+4043455]
	GetHandleVerifier [0x00007FF669FC9756+706710]
	(No symbol) [0x00007FF669EAB8FF]
	(No symbol) [0x00007FF669EA6AE4]
	(No symbol) [0x00007FF669EA6C3C]
	(No symbol) [0x00007FF669E968F4]
	BaseThreadInitThunk [0x00007FFC22F6257D+29]
	RtlUserThreadStart [0x00007FFC23DAAA58+40]


In [46]:
# send the input to the webpage
input_search.send_keys("cotton half sleeve")
sleep(1)
search_button.click()

In [47]:
products = []

for i in range(6):
    print('Scraping page', i+1)
    
    # Scraping product title
    titles = browser.find_elements(By.XPATH, "//span[@class='a-size-large product-title-word-break']")
    title_texts = [title.text.strip() for title in titles]
    
    # Scraping product prices
    prices = browser.find_elements(By.XPATH, "//span[@class='a-price-whole']")
    price_values = [price.text.strip() for price in prices]
    
    # Scraping reviews
    reviews = browser.find_elements(By.XPATH, "//h3[@data-hook='dp-local-reviews-header']")
    review_texts = [review.text.strip() for review in reviews]
    
    # Adding scraped data to the products list
    for title, price, review in zip(title_texts, price_values, review_texts):
        products.append({"Title": title, "Price": price, "Reviews": review})
    
    next_button = browser.find_element(By.XPATH, "//a[@class='s-pagination-item s-pagination-next s-pagination-button s-pagination-separator']")
    next_button.click()
    sleep(5)

Scraping page 1
Scraping page 2
Scraping page 3
Scraping page 4
Scraping page 5
Scraping page 6


In [48]:
# Print the scraped product data
for product in products:
    print(product)

In [49]:
browser.quit()